# Notebook 2a: CV Fold Comparison

---

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Anthropogenic Impact with eDNA and Supervised Machine Learning: Species‑Level Associations from Solomon Island Rivers

---

## Overview
This notebook compares 10-fold vs 5-fold stratified cross-validation to justify the choice of 5-fold CV for model training. Due to limited sample size (n=47), 10-fold CV produces unstable performance estimates with standard deviations often exceeding 0.20 for AUC and accuracy. 5-fold CV increases test-fold sizes to 9-10 sites and reduces variance, providing more reliable performance estimates.

**Purpose**:
- 10-fold vs 5-fold stratified cross-validation comparison.

**Methods:**
- Classifiers: Decision Tree, Random Forest, XGBoost, Logistic Regression.
- Cross-validation: Stratified 10-fold (default parameters).
- Metrics: AUC, Accuracy with fold-level variance.

**Paper Reference**: 2. Materials & Methods -> 2.4 Machine Learning Classification, 2.5 Model Selection and Hyperparameter Tuning; 3. Results -> 3.1 Classifier Performance and Selection

---

## 1. Setup

In [3]:
# Import libraries
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import pickle

# Create output directories
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/performance', exist_ok=True)

print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'Xgboost version: {xgboost.__version__}')
print(f'Matplotlib version: {matplotlib.__version__}')
print(f'Seaborn version: {sns.__version__}')

NumPy version: 1.24.4
Pandas version: 2.3.2
Scikit-learn version: 1.2.2
Xgboost version: 3.0.3
Matplotlib version: 3.9.2
Seaborn version: 0.12.2


## 2. Load Preprocessed Datasets
Load the 8 preprocessed pickle files (4 buffers × 2 thresholds) from Notebook 1 containing X_raw, y, feature_names, and metadata.

**Note**: Raw eDNA counts loaded. Transformations occur within ML pipelines. For rationale see Notebook 1.

In [5]:
# Configuration: 8 datasets (4 buffers × 2 thresholds)
buffers = [100, 300, 500, 1000]
thresholds = [0.03125, 0.0625]

datasets = {}
for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')
        filename = f'data_{buf}m_t{thresh_str}.pkl'
        filepath = os.path.join('results/preprocessed', filename)
        
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        
        key = f'{buf}m_t{thresh_str}'
        datasets[key] = data
        
        print(f'{key}: {data["n_reference"]} reference, {data["n_impacted"]} impacted')

print(f'\nLoaded {len(datasets)} datasets')
print(f'Features per dataset: {len(datasets["100m_t003125"]["feature_names"])}')

100m_t003125: 26 reference, 21 impacted
100m_t00625: 27 reference, 20 impacted
300m_t003125: 22 reference, 25 impacted
300m_t00625: 25 reference, 22 impacted
500m_t003125: 16 reference, 31 impacted
500m_t00625: 23 reference, 24 impacted
1000m_t003125: 18 reference, 29 impacted
1000m_t00625: 29 reference, 18 impacted

Loaded 8 datasets
Features per dataset: 861


## 3. Define ML Pipeline and Classifiers

Create consistent pipeline across all classifiers to ensure fair comparison.
- Classifiers: Decision Tree, Random Forest, XGBoost, Logistic Regression.
- CV Strategy: 10-fold CV (see Notebook 2 for 5-fold comparison, the final selection).

In [7]:
# Define log1p transformer
log_transformer = FunctionTransformer(np.log1p, validate=True)

# 10-fold stratified cross-validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(random_state=42, n_jobs=-1)
}

print(f'Classifiers: {list(classifiers.keys())}')
print(f'CV strategy: 10-fold stratified')

Classifiers: ['Decision Tree', 'Random Forest', 'XGBoost', 'Logistic Regression']
CV strategy: 10-fold stratified


## 4. Classifier Benchmarking: 10-Fold CV with Default Parameters

Initial performance assessment using 10-fold stratified cross-validation with default hyperparameters.

In [9]:
results_10fold = {}

for dataset_key, data in datasets.items():
    print(f'\n{"="*80}')
    print(f'Dataset: {dataset_key}')
    print(f'{"="*80}')
    
    X_raw = data['X_raw']  # numpy array
    y = data['y']  # numpy array
    feature_names = data['feature_names']
    
    results_10fold[dataset_key] = {}
    
    for clf_name, clf in classifiers.items():
        print(f"\nClassifier: {clf_name}")
        
        pipe = Pipeline([
            ('log_transform', log_transformer),
            ('scaler', StandardScaler()),
            ('clf', clf)
        ])

        preds = np.zeros_like(y, dtype=int)
        accuracy_scores = []
        auc_scores = []
        feature_importances = []
        train_accuracy_scores = []
        train_auc_scores = []

        for fold_idx, (train_idx, test_idx) in enumerate(cv_strategy.split(X_raw, y)):
            X_train, X_test = X_raw[train_idx], X_raw[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            pipe.fit(X_train, y_train)

            y_pred = pipe.predict(X_test)
            if hasattr(pipe.named_steps['clf'], "predict_proba"):
                y_proba = pipe.predict_proba(X_test)[:, 1]
                y_train_proba = pipe.predict_proba(X_train)[:, 1]
            else:
                y_proba = pipe.decision_function(X_test)
                y_train_proba = pipe.decision_function(X_train)

            y_train_pred = pipe.predict(X_train)
            preds[test_idx] = y_pred

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            try:
                auc_scores.append(roc_auc_score(y_test, y_proba))
                train_auc_scores.append(roc_auc_score(y_train, y_train_proba))
            except Exception:
                auc_scores.append(np.nan)
                train_auc_scores.append(np.nan)
            
            train_accuracy_scores.append(accuracy_score(y_train, y_train_pred))

            # Feature importances (if available)
            if hasattr(pipe.named_steps['clf'], 'feature_importances_'):
                importances = pipe.named_steps['clf'].feature_importances_
                feature_importances.append(importances)

        # Print summary statistics
        print(f"Mean Train Accuracy: {np.mean(train_accuracy_scores):.4f}, Std: {np.std(train_accuracy_scores):.4f}")
        print(f"Mean Train AUC-ROC: {np.nanmean(train_auc_scores):.4f}, Std: {np.nanstd(train_auc_scores):.4f}")
        print(f"Mean Validation Accuracy: {np.mean(accuracy_scores):.4f}, Std: {np.std(accuracy_scores):.4f}")
        print(f"Mean Validation AUC-ROC: {np.nanmean(auc_scores):.4f}, Std: {np.nanstd(auc_scores):.4f}")
        
        # Classification report
        print(f"Classification report:")
        print(classification_report(y, preds))

        # Store results
        results_10fold[dataset_key][clf_name] = {
            'train_accuracy': train_accuracy_scores,
            'test_accuracy': accuracy_scores,
            'train_auc': train_auc_scores,
            'test_auc': auc_scores,
            'predictions': preds,
            'feature_importances': feature_importances
        }

print("\n10-fold CV benchmarking complete")


Dataset: 100m_t003125

Classifier: Decision Tree
Mean Train Accuracy: 1.0000, Std: 0.0000
Mean Train AUC-ROC: 1.0000, Std: 0.0000
Mean Validation Accuracy: 0.4850, Std: 0.1450
Mean Validation AUC-ROC: 0.4750, Std: 0.1493
Classification report:
              precision    recall  f1-score   support

           0       0.54      0.54      0.54        26
           1       0.43      0.43      0.43        21

    accuracy                           0.49        47
   macro avg       0.48      0.48      0.48        47
weighted avg       0.49      0.49      0.49        47


Classifier: Random Forest
Mean Train Accuracy: 1.0000, Std: 0.0000
Mean Train AUC-ROC: 1.0000, Std: 0.0000
Mean Validation Accuracy: 0.5850, Std: 0.1950
Mean Validation AUC-ROC: 0.6167, Std: 0.2915
Classification report:
              precision    recall  f1-score   support

           0       0.61      0.77      0.68        26
           1       0.57      0.38      0.46        21

    accuracy                           0.6

### 4.1 Create Summary Tables and Export Results

Compile results into summary table focusing on AUC and Accuracy.

In [11]:
summary_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key to extract buffer and threshold
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_10fold[dataset_key][clf_name]
        
        summary_rows.append({
            'Buffer (m)': buffer,
            'Threshold': threshold,
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'Train Accuracy': np.mean(result['train_accuracy']),
            'Train Accuracy Std': np.std(result['train_accuracy']),
            'Test Accuracy': np.mean(result['test_accuracy']),
            'Test Accuracy Std': np.std(result['test_accuracy']),
            'Train AUC': np.nanmean(result['train_auc']),
            'Train AUC Std': np.nanstd(result['train_auc']),
            'Test AUC': np.nanmean(result['test_auc']),
            'Test AUC Std': np.nanstd(result['test_auc'])
        })

summary_df = pd.DataFrame(summary_rows)

# Display test performance only
print("\n10-Fold CV Performance Summary (Test Set):")
print("="*90)
test_cols = ['Buffer (m)', 'Threshold', 'Classifier', 'Test Accuracy', 'Test Accuracy Std', 
             'Test AUC', 'Test AUC Std']
print(summary_df[test_cols].to_string(index=False))

# Save full table to Excel (includes both train and test)
summary_df.to_excel('results/model_performance/10fold_cv_summary.xlsx', index=False)
print("\n" + "="*90)
print("Saved: results/model_performance/10fold_cv_summary.xlsx")


10-Fold CV Performance Summary (Test Set):
 Buffer (m)  Threshold          Classifier  Test Accuracy  Test Accuracy Std  Test AUC  Test AUC Std
        100   0.003125       Decision Tree          0.485           0.145000  0.475000      0.149304
        100   0.003125       Random Forest          0.585           0.195000  0.616667      0.291548
        100   0.003125             XGBoost          0.700           0.165831  0.700000      0.321023
        100   0.003125 Logistic Regression          0.505           0.170953  0.625000      0.247908
        100   0.006250       Decision Tree          0.570           0.241039  0.550000      0.242097
        100   0.006250       Random Forest          0.605           0.233934  0.608333      0.320156
        100   0.006250             XGBoost          0.635           0.225887  0.700000      0.269258
        100   0.006250 Logistic Regression          0.610           0.198494  0.650000      0.280872
        300   0.003125       Decision Tree     

### 4.2 Export Class-Wise Metrics
Extract class-specific performance metrics (precision, recall, F1) for each site class (0=Reference, 1=Impacted) to assess model performance on each class independently, particularly important given class imbalance.

In [13]:
classwise_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_10fold[dataset_key][clf_name]
        
        # Get predictions
        y = datasets[dataset_key]['y']
        preds = result['predictions']
        
        # Calculate class-wise metrics
        from sklearn.metrics import precision_recall_fscore_support
        precision, recall, f1, support = precision_recall_fscore_support(
            y, preds, average=None, zero_division=0
        )
        
        # Add rows for each class
        for class_idx in [0, 1]:
            class_label = 'Reference' if class_idx == 0 else 'Impacted'
            
            classwise_rows.append({
                'Buffer (m)': buffer,
                'Threshold': threshold,
                'Dataset': dataset_key,
                'Classifier': clf_name,
                'Class': class_idx,
                'Class_Label': class_label,
                'Precision': precision[class_idx],
                'Recall': recall[class_idx],
                'F1-Score': f1[class_idx],
                'Support': support[class_idx],
                'Mean_Test_Accuracy': np.mean(result['test_accuracy']),
                'Mean_Test_AUC': np.nanmean(result['test_auc'])
            })

classwise_df = pd.DataFrame(classwise_rows)

# Display format
print("\n" + "="*100)
print("Class-wise Performance Metrics (10-Fold CV):")
print("="*100)

for dataset_key in datasets.keys():
    print(f"\n{dataset_key}:")
    print("-" * 90)
    
    dataset_data = classwise_df[classwise_df['Dataset'] == dataset_key]
    
    for clf_name in classifiers.keys():
        clf_data = dataset_data[dataset_data['Classifier'] == clf_name]
        if len(clf_data) > 0:
            print(f"\n  {clf_name}:")
            print(f"    {'Class':<6} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
            print(f"    {'-'*52}")
            for _, row in clf_data.iterrows():
                print(f"    {row['Class']:<6} {row['Precision']:<12.2f} "
                      f"{row['Recall']:<12.2f} {row['F1-Score']:<12.2f} {row['Support']:<10.0f}")
            
            # Print mean metrics
            mean_acc = clf_data.iloc[0]['Mean_Test_Accuracy']
            mean_auc = clf_data.iloc[0]['Mean_Test_AUC']
            print(f"    Mean Test Accuracy: {mean_acc:.4f}")
            print(f"    Mean Test AUC: {mean_auc:.4f}")

# Save to Excel
classwise_df.to_excel('results/model_performance/10fold_cv_classwise_metrics.xlsx', index=False)
print("\n\nSaved: results/model_performance/10fold_cv_classwise_metrics.xlsx")


Class-wise Performance Metrics (10-Fold CV):

100m_t003125:
------------------------------------------------------------------------------------------

  Decision Tree:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.54         0.54         0.54         26        
    1      0.43         0.43         0.43         21        
    Mean Test Accuracy: 0.4850
    Mean Test AUC: 0.4750

  Random Forest:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.61         0.77         0.68         26        
    1      0.57         0.38         0.46         21        
    Mean Test Accuracy: 0.5850
    Mean Test AUC: 0.6167

  XGBoost:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.73         0.73         0.73         26        
    1      0.67         0.6

### 4.3 Variance Analysis

Quantify 10-fold CV stability to support the methodological decision to adopt 5-fold CV in Notebook 2.

In [15]:
# Extract variance metrics for comparison
variance_comparison = []

for dataset_key in datasets.keys():
    for clf_name in classifiers.keys():
        result = results_10fold[dataset_key][clf_name]
        
        variance_comparison.append({
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'CV_Folds': 10,
            'Acc_Mean': np.mean(result['test_accuracy']),
            'Acc_Std': np.std(result['test_accuracy']),
            'AUC_Mean': np.nanmean(result['test_auc']),
            'AUC_Std': np.nanstd(result['test_auc'])
        })

variance_df = pd.DataFrame(variance_comparison)

# Summary statistics
print("\n" + "="*90)
print("Variance Analysis: 10-Fold CV Stability")
print("="*90)
print(f"\nMean AUC Std across all configurations: {variance_df['AUC_Std'].mean():.4f}")
print(f"Mean Accuracy Std across all configurations: {variance_df['Acc_Std'].mean():.4f}")
print(f"\nConfigurations with AUC Std > 0.20: {(variance_df['AUC_Std'] > 0.20).sum()}/{len(variance_df)}")
print(f"Configurations with Accuracy Std > 0.20: {(variance_df['Acc_Std'] > 0.20).sum()}/{len(variance_df)}")

# Save to Excel
variance_df.to_excel('results/model_performance/10fold_variance_analysis.xlsx', index=False)
print("\nSaved: results/model_performance/10fold_variance_analysis.xlsx")


Variance Analysis: 10-Fold CV Stability

Mean AUC Std across all configurations: 0.1971
Mean Accuracy Std across all configurations: 0.1801

Configurations with AUC Std > 0.20: 14/32
Configurations with Accuracy Std > 0.20: 11/32

Saved: results/model_performance/10fold_variance_analysis.xlsx


### 4.4 Direct Variance Comparison (10-Fold vs 5-Fold)

Compare the stability of 10-fold vs 5-fold CV by examining the standard deviation of performance metrics across folds. Lower standard deviation indicates more stable and reliable performance estimates.

In [17]:
# Load 5-fold variance results from Notebook 2
variance_5fold = pd.read_excel('results/model_performance/5fold_variance_analysis.xlsx')

# 10-fold variance (already computed)
variance_10fold = variance_df  # from previous section

# Combine for comparison
comparison = pd.DataFrame({
    'Metric': ['Mean AUC Std', 'Mean Acc Std', 
               'AUC Std > 0.20 (count)', 'Acc Std > 0.20 (count)'],
    '10-Fold CV': [
        variance_10fold['AUC_Std'].mean(),
        variance_10fold['Acc_Std'].mean(),
        (variance_10fold['AUC_Std'] > 0.20).sum(),
        (variance_10fold['Acc_Std'] > 0.20).sum()
    ],
    '5-Fold CV': [
        variance_5fold['AUC_Std'].mean(),
        variance_5fold['Acc_Std'].mean(),
        (variance_5fold['AUC_Std'] > 0.20).sum(),
        (variance_5fold['Acc_Std'] > 0.20).sum()
    ]
})

# Calculate improvement
comparison['Improvement'] = comparison['10-Fold CV'] - comparison['5-Fold CV']
comparison['% Reduction'] = (comparison['Improvement'] / comparison['10-Fold CV'] * 100).round(1)

print("\n" + "="*90)
print("Variance Comparison: 10-Fold vs 5-Fold CV")
print("="*90)
print(comparison.to_string(index=False))

# Classifier-level comparison
print("\n" + "="*90)
print("Variance Reduction by Classifier:")
print("="*90)

for clf_name in classifiers.keys():
    auc_10 = variance_10fold[variance_10fold['Classifier'] == clf_name]['AUC_Std'].mean()
    auc_5 = variance_5fold[variance_5fold['Classifier'] == clf_name]['AUC_Std'].mean()
    reduction = ((auc_10 - auc_5) / auc_10 * 100)
    
    print(f"{clf_name}:")
    print(f"  10-fold: {auc_10:.4f} ± AUC | 5-fold: {auc_5:.4f} ± AUC")
    print(f"  Variance reduction: {reduction:.1f}%")


Variance Comparison: 10-Fold vs 5-Fold CV
                Metric  10-Fold CV  5-Fold CV  Improvement  % Reduction
          Mean AUC Std    0.197098   0.138025     0.059074         30.0
          Mean Acc Std    0.180137   0.120894     0.059243         32.9
AUC Std > 0.20 (count)   14.000000   3.000000    11.000000         78.6
Acc Std > 0.20 (count)   11.000000   0.000000    11.000000        100.0

Variance Reduction by Classifier:
Decision Tree:
  10-fold: 0.1832 ± AUC | 5-fold: 0.1320 ± AUC
  Variance reduction: 27.9%
Random Forest:
  10-fold: 0.2045 ± AUC | 5-fold: 0.1421 ± AUC
  Variance reduction: 30.5%
XGBoost:
  10-fold: 0.2063 ± AUC | 5-fold: 0.1562 ± AUC
  Variance reduction: 24.3%
Logistic Regression:
  10-fold: 0.1944 ± AUC | 5-fold: 0.1218 ± AUC
  Variance reduction: 37.3%


## 5. Summary

**Outputs** (`results/model_performance/`):
- `10fold_cv_summary.xlsx` 
- `10fold_cv_classwise_metrics.xlsx` 
- `10fold_variance_analysis.xlsx` 

## Next Steps
Return to Notebook 2 (`2_Model_Training.ipynb`) for the full modelling pipeline using 5-fold CV.